# ಪಾಠ 12 - ಏಜೆಂಟ್ ಸ್ಕೆಚ್‌ಪ್ಯಾಡ್‌ನೊಂದಿಗೆ ಚಾಟ್ ಇತಿಹಾಸ ಕಡಿತ

ಈ ನೋಟ್ಬುಕ್ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿಕೊಂಡು ದೀರ್ಘ ಸಂಭಾಷಣೆಗಳಲ್ಲಿ ಸತ್ಯಾಂಶವನ್ನು ಹೇಗೆ ನಿರ್ವಹಿಸುವುದೆಂಬುದನ್ನು ತೋರಿಸುತ್ತದೆ. ಸಂಭಾಷಣೆಗಳು ವೃದ್ಧಿಯಾಗುವಂತೆ ಟೋಕನ್ ಸಂಖ್ಯೆಯೂ ಹೆಚ್ಚಾಗುತ್ತದೆ — ಕೊನೆಗೆ ಮಾದರಿಯ ಸತ್ಯಾಂಶ ವಿಂಡೋವನ್ನು ಮೀರಿಸುತ್ತದೆ. ಇದನ್ನು ನಾವು **ಸಂದರ್ಭಸಂಗ್ರಹಣ ಮಾದರಿಯ** ಮೂಲಕ ಮತ್ತು **ಏಜೆಂಟ್ ಸ್ಕೆಚ್‌ಪ್ಯಾಡ್** ಮೂಲಕ ಶಾಶ್ವತ ಸ್ಮರಣೆಯನ್ನು ನಿರ್ವಹಿಸುವ ಮೂಲಕ ಪರಿಹರಿಸುತ್ತೇವೆ.

## ನೀವು ಕಲಿಯುವವುಗಳು:
1. **ಸಂದರ್ಭ ನಿರ್ವಹಣೆ ಏಕೆ ಮಹತ್ವದದ್ದು**: ಟೋಕನ್ ಮಿತಿಗಳು ಮತ್ತು ಸತ್ಯಾಂಶ ವಿಂಡೋಗಳನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು
2. **ಸಂದರ್ಭ-ಜಾಗೃತಿ ಏಜೆಂಟ್‌ಗಳು**: ಅವರದೇ ಸಂಭಾಷಣೆಯ ಸತ್ಯಾಂಶವನ್ನು ನಿರ್ವಹಿಸುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸುವುದು
3. **ಸಂದರ್ಭಸಂಗ್ರಹಣ ಮಾದರಿ**: ಸಂಭಾಷಣಾ ಇತಿಹಾಸವನ್ನು ಸಂಕ್ಷಿಪ್ತಗೊಳಿಸಲು ಉಪಕರಣಗಳನ್ನು ಬಳಸುವುದು
4. **ಏಜೆಂಟ್ ಸ್ಕೆಚ್‌ಪ್ಯಾಡ್**: ಸತ್ಯಾಂಶ ಕಡಿತದ ಜೊತೆಗೆ ಉಳಿಯುವ ಶಾಶ್ವತ ಸ್ಮರಣೆ

## ಅಗತ್ಯವಿರುವ ಪೂರ್ವನಿಬಂಧನೆಗಳು:
- ಪರಿಸರ ಬದಲಾವಣೆಯೊಂದಿಗೆ ಅಝೂರ್ ಓಪನ್‌ಎಐ ವ್ಯವಸ್ಥೆ ಸ್ಥಾಪನೆ
- ಹಿಂದಿನ ಪಾಠಗಳಿಂದ ಮೂಲ ಏಜೆಂಟ್ ಪರಿಕಲ್ಪನೆಗಳ ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವಿಕೆ


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ಸಂದರ್ಭ ನಿರ್ವಹಣೆ ಮಹತ್ತರದ ಕಾರಣ

ಪ್ರತಿಯೊಂದು LLMಗೆ ಸೀಮಿತ **ಪರಿಸ್ಥಿತಿ ವಿಂಡೋ** ಇದೆ — ಇದು ಒಂದು ವಿನಂತಿಯಲ್ಲಿ ಪ್ರಕ್ರಿಯೆ ಮಾಡಬಹುದಾದ ಗರಿಷ್ಠ ಟೋಕನ್‌ಗಳ ಸಂಖ್ಯೆ. ಬಹು-ಮೊದಲು ಸಂಭಾಷಣೆ ಮುಂದುವರಿದಂತೆ:

- **ಟೋಕನ್ ಎಣಿಕೆ ಲೀನಿಯರ್ ರೀತಿಯಲ್ಲಿ** ಪ್ರತಿಯೊಂದು ಬಳಕೆದಾರ ಸಂದೇಶ ಮತ್ತು ಸಹಾಯಕ ಪ್ರತಿಕ್ರಿಯೆಯೊಂದಿಗೆ ಹೆಚ್ಚಾಗುತ್ತದೆ.
- **ಪ್ರಾಂಪ್ಟ್ ಟೋಕನ್‌ಗಳು ವೆಚ್ಚವನ್ನು ಪ್ರಭಾವಿಸುತ್ತವೆ** ಏಕೆಂದರೆ ಸಂಪೂರ್ಣ ಇತಿಹಾಸವನ್ನು ಪ್ರತಿಯೊಂದು ತಿರುಗಾಟದಲ್ಲಿಯೂ ಮರುಕಳುಹಿಸಲಾಗುತ್ತದೆ.
- ಕೊನೆಯಲ್ಲಿ ಸಂಭಾಷಣೆ **ಪರಿಸ್ಥಿತಿ ವಿಂಡೋವನ್ನು ಮೀರುತ್ತದೆ** ಮತ್ತು ಮಾದರಿ either ಸಣ್ಣಮಾಡುತ್ತದೆ ಅಥವಾ ದೋಷವನ್ನು ನೀಡುತ್ತದೆ.

### ಸಂದರ್ಭ ನಿರ್ವಹಣೆಗಾಗಿ ಹೆಣಕು ನೀತಿಗಳು

| ಹದಿನೆಯಾಗಿಸುವ ವಿಧಾನ | ಹೇಗೆ ಕೆಲಸಮಾಡುತ್ತದೆ | ಹಾನಿ-ಫಲಿತಾಂಶ |
|---|---|---|
| **ಸಣ್ಣಮಾಡುವುದು** | ಹಳೆಯ ಸಂದೇಶಗಳನ್ನು ತೆಗೆದುಹಾಕುವುದು | ಮೊದಲಿನ ಪರಿಸ್ಥಿತಿಯನ್ನು ಕಳೆದುಕೊಳ್ಳುವುದು |
| **ಸಾರಾಂಶ ರಚನೆ** | ಹಳೆಯ ಸಂದೇಶಗಳನ್ನು ಸಾರಾಂಶವಾಗಿ ಸಂಕ್ಷಿಪ್ತಗೊಳಿಸುವುದು | ಕೆಲವು ವಿವರಗಳು ಕಳೆದುಹೋಗುತ್ತವೆ, ಆದರೆ ಪ್ರಮುಖ ಅಂಶಗಳು ಉಳಿಯೋದು |
| **ಸ್ಕ್ರ್ಯಾಚ್‌ಪ್ಯಾಡ್ / ಬಾಹ್ಯ ಮೆಮೊರಿ** | ಸಂಭಾಷಣೆಗೆ ಹೊರಗಿನ ಮುಖ್ಯ ಮಾಹಿತಿಗಳನ್ನು ಸಂಗ್ರಹಿಸುವುದು | ಸಾಧನ ಕರೆದೊಯ್ಯಲು ಅಗತ್ಯವಿದೆ, ಆದರೆ ಯಾವುದೇ ಕುಸಿತಕ್ಕೂ ತಾಳುವ ಶಕ್ತಿ ಇರುವುದು |

ಈ ನೋಟ್‌ಬುಕ್‌ನಲ್ಲಿ ನಾವು **ಸಾರಾಂಶ ರಚನೆ** ಮತ್ತು **ಸ್ಕ್ರ್ಯಾಚ್‌ಪ್ಯಾಡ್ ಉಪಕರಣ** ಅನ್ನು ಜೋಡಿಸುತ್ತೇವೆ ಆದಕಾರಣ ಏಜೆಂಟ್ ಸಂಭಾಷಣೆ ಇತಿಹಾಸವನ್ನು ಸಂಕ್ಷಿಪ್ತಗೊಳಿಸಿದಾಗಲೂ ನಿಯತತೆ ಕಾಪಾಡಿಕೊಳ್ಳಬಹುದು.


## సందర్భ-ಜಾಗರೂಕ ಏಜೆಂಟ್ ರಚನೆ


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## ದೀರ್ಘ ಸಂವಾದವನ್ನು ಅನುಕರಿಸುವುದು

ಪ್ರಾಯೋಗಿಕವಾಗಿ ಒಂದಕ್ಕಿಂತ ಹೆಚ್ಚು ಸಮಯ ಮುಂದುವರಿಯುವ ಸಂವಾದವನ್ನು ನೋಡೋಣ, ಇದರಿಂದ ಪ್ರ ಸ್ಫೂರ್ತಿಯು ಹೇಗೆ ಸಂಗ್ರಹಿಸಲಾಗುತ್ತದೆ ಎಂದು ತಿಳಿದುಕೊಳ್ಳೋಣ. ಪ್ರತಿನಿಧಿಯು ಪ್ರಮುಖ ವಿವರಗಳನ್ನು (ಆಸಕ್ತಿಗಳು, ಬಜೆಟ್, ಪ್ರಯಾಣ ದಿನಾಂಕಗಳು) ಸಂವಾದದ ನಡುವೆ ಹಿಡಿದುಕೊಂಡು ನಿರಂತರತೆಯನ್ನು ತೋರಿಸಬೇಕು.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ಏಜೆಂಟ್ ಹಿಂದಿನ ತಿರುವುಗಳಿಂದ ಕಂಡು ಹಿಡಿದಿರುವ ಸੰਦਰಭವನ್ನು ಗಮನಿಸಿ — ಅದು ಜಪಾನಿನ ಬಗ್ಗೆ, ಸುಶಿ, ದೇವಾಲಯಗಳು, ಚಿತ್ರಕಲೆ, $3000 ಬಜೆಟ್, ಏಕಾಂಗಿ ಪ್ರಯಾಣ, ಮತ್ತು ಏಪ್ರಿಲ್ ಸಮಯದ ಬಗ್ಗೆ ತಿಳಿಯಿತು. ಒಂದು ಚಿಕ್ಕ ಮಾತುಕಥೆಯಲ್ಲಿ ಇದು ಚೆನ್ನಾಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ, ಆದರೆ ಮಾತುಕಥೆ ವೃದ್ಧಿಯಾಗುತ್ತಿದ್ದಂತೆ ಸಂಪೂರ್ಣ ಇತಿಹಾಸವನ್ನು ಮರು-ಕಳುಹಿಸುವುದು ದುಬಾರಿ ಆಗುತ್ತದೆ.

ಸੰਦਰಭ ಸಂಗ್ರಹಣೆಯನ್ನು ನೋಡಲು ಹೆಚ್ಚು ತಿರುವುಗಳೊಂದಿಗೆ ಮಾತುಕಥೆಯನ್ನು ಮುಂದುವರೆಸೋಣ:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## ಸಂದರ್ಭ ಸಂಕ್ಷಿಪ್ತಣ ಮಾದರಿ

ಸಂಭಾಷಣೆ ವೃದ್ಧಿಯಾಗುತ್ತಿರುವಂತೆ, ನಾವು ಸಂಗ್ರಹಿತ ಸಂದರ್ಭವನ್ನುಸಂಚಿಪ್ತ ಸ್ವರೂಪದಲ್ಲಿ ಒಗ್ಗೂಡಿಸಲು **ಸಂಕ್ಷಿಪ್ತಣ ಸಲಕರಣೆ** ಬಳಸಬಹುದು. ಏಜೆಂಟ್ ಈ ಸಲಕರಣೆಯನ್ನು ಕರೆಸಿ ಮುಖ್ಯ ಆದ್ಯತೆಗಳನ್ನು ದಾಖಲು ಮಾಡುತ್ತದೆ, ಹೀಗಾಗಿ ಹಳೆಯ ಸಂದೇಶಗಳನ್ನು ತೆಗೆದುಹಾಕಿದರೂ ಸಹ, ಅಗತ್ಯವಿರುವ ಮಾಹಿತಿಯನ್ನು ಉಳಿಸಬಹುದಾಗುತ್ತದೆ.

ಈ ಮಾದರಿ ಹೆಚ್ಚು ತಾಂತ್ರಿಕ ಇತಿಹಾಸ ಕಡಿತಗೊಳಿಸುವಿಕೆಯ ನಿರ್ಮಾಣ ಖಂಡವಾಗಿದೆ:
1. ಏಜೆಂಟ್ ಸಂಭಾಷಣೆಯಿಂದ ಮುಖ್ಯ ಸತ್ಯಗಳನ್ನು ಗುರುತಿಸುತ್ತದೆ
2. ಅದನ್ನು ಸಂಕ್ಷಿಪ್ತಣ ಸಲಕರಣೆಗೆ ಕರೆಸುತ್ತದೆ ಅದನ್ನು ಸ್ಥಿರಪಡಿಸಲು
3. ಹಳೆಯ ಸಂದೇಶಗಳನ್ನು ಸುರಕ್ಷಿತವಾಗಿ ತೆಗೆದುಹಾಕಬಹುದು ಏಕೆಂದರೆ ಸಂಕ್ಷಿಪ್ತಣೆ ಮುಖ್ಯ ವಿಷಯವನ್ನು ಹಿಡಿದಿಡುತ್ತದೆ

ಕೆಳಗಿನಲ್ಲಿ ನಾವು ಒಂದು `summarize_preferences` ಸಲಕರಣೆ ನಿರ್ದಿಷ್ಟಮಾಡಿದ್ದೇವೆ, ಇದು ಏಜೆಂಟ್ ಕಲಿತ ಸಂಗತಿಗಳ ಸಂಕ್ಷಿಪ್ತದ ಪಟ್ಟಿಯನ್ನು ದಾಖಲಿಸಲು ಕರೆಸಬಹುದು.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ಬಳಸಿ ದೀರ್ಘಕಾಲಿಕ ಏಜೆಂಟ್ ಸಂಭಾಷಣೆಗಳಲ್ಲಿ ಪ್ರಸಂಗವನ್ನು ಹೇಗೆ ನಿರ್ವಹಿಸುವುದೆಂಬುದನ್ನು ಕಲಿತಿರಿ:

### ಪ್ರಮುಖ ಪರಿಕಲ್ಪನೆಗಳು
- **ಪ್ರಸಂಗ ಕಿಟಕಿಗಳು ಸೀಮಿತವಾಗಿವೆ** — ಸಂಭಾಷಣಾ ಇತಿಹಾಸದಲ್ಲಿನ ಪ್ರತಿ ಟೋಕನ್‌ಗೆ ಖರ್ಚು ಮತ್ತು ಮಿತಿ ಲಾಗುತ್ತದೆ.
- **ಸಾರಾಂಶಕರಣ ಸಾಧನಗಳು** ಏಜೆಂಟ್ ಅನ್ನು ಸಂಗ್ರಹಿತ ಪ್ರಸಂಗವನ್ನು ಸಂಕ್ಷಿಪ್ತ ಸಾರಾಂಶಗಳಾಗಿ ರೂಪಿಸಲು ಅನುಮತಿಸುತ್ತವೆ, ಆದ್ದರಿಂದ ಟೋಕನ್ ಬಳಕೆ ಕಡಿಮೆಯಾಗುತ್ತಾ ಅಗತ್ಯ ಮಾಹಿತಿಯನ್ನು ಕಾಯ್ದುಕೊಳ್ಳುತ್ತವೆ.
- **ಏಜೆಂಟ್ ಸ್ಕ್ರಾಚ್‌ಪ್ಯಾಡ್ಗಳು** ಯಾವುದೇ ಸಂಭಾಷಣೆ ಕಡಿತನಂತರವೂ ಶಾಶ್ವತ ಬಾಹ್ಯ ಸ್ಮರಣೆಯನ್ನು ಒದಗಿಸುತ್ತವೆ.

### ನೀವು ಏನು ನಿರ್ಮಿಸಿದ್ದೀರಿ
- ಬಹುಮನೋನಿತ ಸಂಭಾಷಣೆಗಳಲ್ಲಿ ನಿರಂತರತೆಯನ್ನು ಕಾಯ್ದುಕೊಳ್ಳುವ **ಪ್ರಸಂಗ-ಜ್ಞಾನಿ ಏಜೆಂಟ್**
- ಪ್ರಮುಖ ಬಳಕೆದಾರ ವಿವರಗಳನ್ನು ಸಂಕ್ಷಿಪ್ತ ಸ್ವರೂಪದಲ್ಲಿ ದಾಖಲಿಸುವ **ಸಾರಾಂಶಕರಣ ಸಾಧನ** (`summarize_preferences`)
- ಪ್ರಸಂಗವು ಉಳಿಸುವ ಹಾಗೂ ಬದಲಾವಣೆಗಳನ್ನು ಕೈಗಾರಿಸುವ **ಬಹು-ತಿರುಗು ಸಂಭಾಷಣೆ**

### ನೈಜ ಜಗತ್ತಿನ ಅನ್ವಯಿಕೆಗಳು
- **ಗ್ರಾಹಕ ಸೇವಾ ಬಾಟ್‌ಗಳು**: ದೀರ್ಘ ಬೆಂಬಲ ಸತ್ರಗಳಲ್ಲಿ ಪ್ರಾಧಾನ್ಯತೆಯನ್ನು ನೆನಪಿಡಿ
- **ವೈಯಕ್ತಿಕ ಸಹಾಯಕರು**: ಪ್ರಸಂಗವನ್ನು ಪುನರ್ ವಿವರಣೆ ಮಾಡದೆ ಮುಂದುವರೆದ ಪ್ರಾಜೆಕ್ಷನ್‌ಗಳನ್ನು ಅನುಸರಿಸು
- **ಶೈಕ್ಷಣಿಕ ಟ್ಯೂಟರ್‌ಗಳು**: ಅನೇಕ ಸಂವಾದಗಳಾದ ನಂತರ ವಿದ್ಯಾರ್ಥಿ ಪ್ರಗತಿಯನ್ನು ನಿರ್ವಹಿಸು

### ಮುಂದಿನ ಹೆಜ್ಜೆಗಳು
- ಫೈಲ್ ಆಧಾರಿತ ಶಾಶ್ವತತೆಯೊಂದಿಗೆ ಸಂಪೂರ್ಣ ಸ್ಕ್ರಾಚ್‌ಪ್ಯಾಡ್ ಸಾಧನವನ್ನು ಅನುಷ್ಟಾನಗೊಳಿಸಿ
- ಸಾರಾಂಶಕರಣದ ನಂತರ ಸ್ವಯಂಚಾಲಿತ ಇತಿಹಾಸ ಕಡಿತವನ್ನು ಸೇರಿಸಿ
- ಅರ್ಥ ಸ್ಮರಣೆ ಹುಡುಕಾಟಕ್ಕಾಗಿ ವೆಕ್ಟರ್ ಡೇಟಾಬೇಸ್‌ಗಳೊಡನೆ ಸಂಯೋಜಿಸಿ
- ಪೂರ್ಣ ಪ್ರಸಂಗದೊಂದಿಗೆ ದಿನಗಳ ನಂತರ ಸಂಭಾಷಣೆಯನ್ನು ಪುನರಾರಂಭಿಸಬಹುದಾದ ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸಿ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
